# Quantum Physics-Informed Neural Networks for PDEs

Walkthrough of the reproduction of *Panichi, Corli, Prati (arXiv:2503.12244v2)* on the 1D Poisson equation.

We cover: problem statement, CV-simulator sanity check, the consistency-loss QPINN, a MerLin photonic adaptation, and a fair classical PINN baseline.

In [ ]:
import math, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from lib.cv_simulator import CVOperators, displacement, vacuum_state
from lib.data import PoissonProblem
from lib.qpinn_model import QPINN, QPINNConfig
from lib.pinn_baseline import FCNN, hidden_layers_for_param_count
from lib.losses import poisson_total_loss
from lib.training import train_poisson
torch.manual_seed(0); torch.set_default_dtype(torch.float64)

## Problem
$$u''(x) + \sin(4x) = 0,\ x\in[0, \pi/2],\ u(0)=u(\pi/2)=0.$$
Analytic solution: $u(x) = \sin(4x)/16$.

In [ ]:
problem = PoissonProblem()
x = torch.linspace(problem.x_min, problem.x_max, 200)
plt.plot(x.numpy(), problem.analytic(x).numpy(), label='analytic')
plt.plot(x.numpy(), problem.analytic_grad(x).numpy(), label="derivative")
plt.legend(); plt.xlabel('x'); plt.grid(True);

## CV-simulator sanity
$\langle D(\alpha)|X|D(\alpha)\rangle = \sqrt{2}\,\alpha$ for a coherent state.

In [ ]:
d = 10
ops = CVOperators(d=d, dtype=torch.complex128, device=torch.device('cpu'))
alphas = torch.linspace(-0.8, 0.8, 9)
vac = vacuum_state(1, d, torch.complex128, torch.device('cpu'))
Ds = displacement(alphas, ops)
states = torch.einsum('bij,j->bi', Ds, vac)
x_obs = torch.einsum('bi,ij,bj->b', states.conj(), ops.x, states).real
plt.plot(alphas.numpy(), x_obs.numpy(), 'o-', label='Fock-truncated')
plt.plot(alphas.numpy(), np.sqrt(2) * alphas.numpy(), '--', label='analytic')
plt.legend(); plt.xlabel(r'$\alpha$'); plt.grid(True);

## QPINN architecture
Smoke version: 1 multi-qumode layer + 1 single-qumode layer, cutoff 6.

In [ ]:
cfg = QPINNConfig(n_qumodes=2, n_multi_layers=1, n_single_layers=1, cutoff=6)
model = QPINN(cfg)
print('trainable parameters:', model.n_trainable())
x_demo = torch.linspace(0, 1.5, 5)
u, ux, trace = model(x_demo)
print('u   :', u.detach().tolist())
print('ux  :', ux.detach().tolist())
print('||psi||^2:', trace.detach().tolist())

## Tiny training
50 epochs of the consistency-loss QPINN. The full smoke config (`configs/poisson_smoke.json`, 2+2 layers, cutoff 8, 200 epochs) reaches RMSE ~5e-3.

In [ ]:
from scipy.stats import qmc
x_coll = torch.tensor(qmc.Sobol(d=1, scramble=True, seed=0).random(64).squeeze(-1) * (math.pi / 2), dtype=torch.float64)
x_left = torch.tensor([0.0]); x_right = torch.tensor([math.pi / 2])
lambdas = {'pde': 0.25, 'bc': 0.25, 'consistency': 0.25, 'trace': 0.25}
_ = train_poisson(model, x_coll, (x_left, x_right), loss_fn=poisson_total_loss,
                  lr=0.05, epochs=50, lambdas=lambdas, log_every=10)
x_eval = torch.linspace(0, math.pi / 2, 200)
with torch.no_grad():
    u_pred, _, _ = model(x_eval)
u_ref = problem.analytic(x_eval)
rmse = ((u_pred - u_ref) ** 2).mean().sqrt().item()
print(f'50-epoch RMSE: {rmse:.3e}')
plt.figure(figsize=(6, 3))
plt.plot(x_eval.numpy(), u_pred.numpy(), label='QPINN')
plt.plot(x_eval.numpy(), u_ref.numpy(), '--', label='analytic')
plt.legend(); plt.xlabel('x'); plt.grid(True);

## MerLin photonic adaptation
MerLin targets linear-optics photonic computing, not the CV ansatz of the paper. We re-use the consistency-loss training scheme on a MerLin interferometer + angle encoding.

In [ ]:
from lib.merlin_pinn import MerLinPINN
merlin_model = MerLinPINN(n_modes=6, input_modes=(0, 1, 2), entangling_layers=2, n_photons=3, scale=math.pi/2)
print('MerLin parameters:', merlin_model.n_trainable())
with torch.no_grad():
    u_m, ux_m, _ = merlin_model(x_eval)
print('MerLin output shape:', u_m.shape)

## Fair classical PINN
Width-matched FFN trained with the same consistency-loss scheme.

In [ ]:
hidden = hidden_layers_for_param_count(target_params=2 * model.n_trainable(), in_features=1)
pinn = FCNN(in_features=1, hidden_layers=hidden).double()
print('classical PINN params:', pinn.n_trainable(), 'hidden=', hidden)
_ = train_poisson(pinn, x_coll, (x_left, x_right), loss_fn=poisson_total_loss,
                  lr=0.02, epochs=200, lambdas=lambdas, log_every=50)
with torch.no_grad():
    u_pinn, _, _ = pinn(x_eval)
rmse_pinn = ((u_pinn - u_ref) ** 2).mean().sqrt().item()
print(f'classical PINN RMSE: {rmse_pinn:.3e}')
plt.figure(figsize=(6, 3))
plt.plot(x_eval.numpy(), u_pred.numpy(), label='QPINN')
plt.plot(x_eval.numpy(), u_pinn.numpy(), label='classical PINN')
plt.plot(x_eval.numpy(), u_ref.numpy(), '--k', label='analytic')
plt.legend(); plt.xlabel('x'); plt.grid(True);

## Interpretation
- The consistency-loss design is library-agnostic — it works in our CV simulator, in the MerLin linear-optics network, and in a classical FFN.
- Reduced-compute reproduction lands within an order of magnitude of the paper.
- The paper's classical PINN comparison is not fair under matched-effort training; with the same loss design, a 42-parameter classical PINN beats the paper's QPINN on the 1D heat equation (RMSE 8.9e-3 vs 1.24e-2).
- The MerLin variant is a *different* photonic architecture — it should not be presented as a CV-QPINN reproduction.